In [3]:
import pefile

In [5]:
pe = pefile.PE("datos\\pavimentos\\libop64.dll")

In [6]:
for exp in pe.DIRECTORY_ENTRY_EXPORT.symbols:
    print(exp.name)

b'OP_HT_Init'
b'OP_HT_Interpolate'
b'OP_HT_Reset'
b'OP_HT_Step'
b'OP_LE_Calc'
b'OP_LE_Calc_CalME'


In [ ]:
from ctypes import cdll, c_long, c_double, POINTER
import numpy as np
import pandas as pd

# === 1️⃣ Cargar la DLL ===
dll_path = "datos\\pavimentos\\libop64.dll"

In [28]:
# -------- Cargar la librería --------
lib = cdll.LoadLibrary(dll_path)

# -------- Firma (la que ya funcionó anteriormente) --------
lib.OP_LE_Calc.argtypes = [
    c_long,  # Flags
    c_long, POINTER(c_double), POINTER(c_double), POINTER(c_double), POINTER(c_double),
    c_long, POINTER(c_double), POINTER(c_double), POINTER(c_double), POINTER(c_double), POINTER(c_double),
    c_long, POINTER(c_double), POINTER(c_double), POINTER(c_double), POINTER(c_long),
    POINTER(c_double)
]
lib.OP_LE_Calc.restype = c_long

# -------- Valores de entrada (idénticos a tu Excel) --------
Flags = c_long(1)  # modo Default
nLayers = c_long(2)

# capas
h = np.array([100.0, 0.0], dtype=np.double)        # thickness mm
E = np.array([1000.0, 100.0], dtype=np.double)     # elastic modulus MPa
v = np.array([0.35, 0.35], dtype=np.double)        # Poisson
f = np.array([1.0, 1.0], dtype=np.double)          # friction

# cargas
nLoads = c_long(1)
Ax = np.array([0.0], dtype=np.double)              # X (mm)
Ay = np.array([0.0], dtype=np.double)              # Y (mm)
Al = np.array([20.0], dtype=np.double)             # Load (kN)
Ap = np.array([710.0], dtype=np.double)            # Pressure (kPa)
Ar = np.array([94.691476], dtype=np.double)        # Radius (mm)

# puntos de evaluación
nPoints = c_long(1)
Px = np.array([0.0], dtype=np.double)
Py = np.array([0.0], dtype=np.double)
Pz = np.array([0.0], dtype=np.double)
Pl = np.array([1], dtype=np.int64)                 # layer index (1-based in Excel)

# arreglo resultado (27 valores por punto)
Result = np.zeros(27 * 1, dtype=np.double)

# -------- Llamada --------
rv = lib.OP_LE_Calc(
    Flags,
    nLayers,
    h.ctypes.data_as(POINTER(c_double)),
    E.ctypes.data_as(POINTER(c_double)),
    v.ctypes.data_as(POINTER(c_double)),
    f.ctypes.data_as(POINTER(c_double)),
    nLoads,
    Ax.ctypes.data_as(POINTER(c_double)),
    Ay.ctypes.data_as(POINTER(c_double)),
    Al.ctypes.data_as(POINTER(c_double)),
    Ap.ctypes.data_as(POINTER(c_double)),
    Ar.ctypes.data_as(POINTER(c_double)),
    nPoints,
    Px.ctypes.data_as(POINTER(c_double)),
    Py.ctypes.data_as(POINTER(c_double)),
    Pz.ctypes.data_as(POINTER(c_double)),
    Pl.ctypes.data_as(POINTER(c_long)),
    Result.ctypes.data_as(POINTER(c_double))
)

# Comprueba retorno
print("Return code:", rv)

# -------- Mapear resultados según el comentario del VBA en el .xlsm --------
# VBA decía:
# 1-6:   sxx, syy, szz, sxy, sxz, syz
# 7-12:  sp1, sp2, sp3, ss1, ss2, ss3   (sp = principal stresses ?, ss = some shear terms)
# 13-15: dxx, dyy, dzz  (deflections)
# 16-21: exx, eyy, ezz, exy, exz, eyz  (strains)
# 22-27: ep1, ep2, ep3, es1, es2, es3  (principal strains + shear principals)

R = Result.reshape((27, 1))[:, 0]  # primer punto

# Extraer por nombre (0-based indices)
sxx = R[0]; syy = R[1]; szz = R[2]
sxy = R[3]; sxz = R[4]; syz = R[5]

sp1 = R[6]; sp2 = R[7]; sp3 = R[8]
ss1 = R[9]; ss2 = R[10]; ss3 = R[11]

dxx = R[12]; dyy = R[13]; dzz = R[14]      # deflexiones (según VBA)
exx = R[15]; eyy = R[16]; ezz = R[17]
exy = R[18]; exz = R[19]; eyz = R[20]

ep1 = R[21]; ep2 = R[22]; ep3 = R[23]
es1 = R[24]; es2 = R[25]; es3 = R[26]

# -------- Convertir a las unidades que muestra el Excel --------
# Observación práctica (coincide con tus datos): multiplicadores para que coincida con Excel
# - Tensiones -> kPa  : multiplicar por 1e6
# - Deformaciones -> με : multiplicar por 1e6
# - Deflexiones están ya en mm según tu Excel (no multiplicar)

# Tensiones (kPa)
sxx_kPa = sxx * 1e6
syy_kPa = syy * 1e6
szz_kPa = szz * 1e6
sxy_kPa = sxy * 1e6
sxz_kPa = sxz * 1e6
syz_kPa = syz * 1e6

sp1_kPa = sp1 * 1e6
sp2_kPa = sp2 * 1e6
sp3_kPa = sp3 * 1e6
ss1_kPa = ss1 * 1e6
ss2_kPa = ss2 * 1e6
ss3_kPa = ss3 * 1e6

# Deformaciones (με)
exx_micro = exx * 1e6
eyy_micro = eyy * 1e6
ezz_micro = ezz * 1e6
exy_micro = exy * 1e6
exz_micro = exz * 1e6
eyz_micro = eyz * 1e6

ep1_micro = ep1 * 1e6
ep2_micro = ep2 * 1e6
ep3_micro = ep3 * 1e6
es1_micro = es1 * 1e6
es2_micro = es2 * 1e6
es3_micro = es3 * 1e6

# Deflexiones (mm) — según tu Excel dzz ya coincide directamente
dx_mm = dxx
dy_mm = dyy
dz_mm = dzz

# -------- Armar DataFrame en el mismo orden que tu Excel --------
ordered = [
    ("δx (mm)", dx_mm),
    ("δy (mm)", dy_mm),
    ("δz (mm)", dz_mm),

    ("sxx (kPa)", sxx_kPa),
    ("syy (kPa)", syy_kPa),
    ("szz (kPa)", szz_kPa),

    ("txy (kPa)", sxy_kPa),
    ("txz (kPa)", sxz_kPa),
    ("tyz (kPa)", syz_kPa),

    ("exx (με)", exx_micro),
    ("eyy (με)", eyy_micro),
    ("ezz (με)", ezz_micro),

    ("gxy (με)", exy_micro),
    ("gxz (με)", exz_micro),
    ("gyz (με)", eyz_micro),

    ("σ1 (kPa)", sp1_kPa),
    ("σ2 (kPa)", sp2_kPa),
    ("σ3 (kPa)", sp3_kPa),

    ("τ1 (kPa)", ss1_kPa),
    ("τ2 (kPa)", ss2_kPa),
    ("τ3 (kPa)", ss3_kPa),

    ("ε1 (με)", ep1_micro),
    ("ε2 (με)", ep2_micro),
    ("ε3 (με)", ep3_micro),

    ("γ1 (με)", es1_micro),
    ("γ2 (με)", es2_micro),
    ("γ3 (με)", es3_micro),
]

df = pd.DataFrame(ordered, columns=["Name", "Value"]).set_index("Name")

# Formato visual similar a Excel (opcional)
pd.set_option("display.float_format", "{:,.6f}".format)

print("\nResultados formateados (mismo orden y unidades que Excel):\n")
print(df)

Return code: 0

Resultados formateados (mismo orden y unidades que Excel):

                  Value
Name                   
δx (mm)        0.000000
δy (mm)        0.000000
δz (mm)        0.000511
sxx (kPa) -1,272.826498
syy (kPa) -1,272.826498
szz (kPa)   -710.000003
txy (kPa)      0.000000
txz (kPa)      0.000000
tyz (kPa)      0.000000
exx (με)      -0.578837
eyy (με)      -0.578837
ezz (με)       0.180979
gxy (με)       0.000000
gxz (με)       0.000000
gyz (με)       0.000000
σ1 (kPa)  -1,272.826498
σ2 (kPa)  -1,272.826498
σ3 (kPa)    -710.000003
τ1 (kPa)       0.000000
τ2 (kPa)     281.413247
τ3 (kPa)     281.413247
ε1 (με)       -0.578837
ε2 (με)       -0.578837
ε3 (με)        0.180979
γ1 (με)        0.000000
γ2 (με)        0.759816
γ3 (με)        0.759816
